<a href="https://colab.research.google.com/github/Shemuel-bot/AI-ball-notebooks/blob/main/ai_ball_st_gs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ai_ball_7.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1RScD6nl_1_teIc-eGZ_Tao-A5BP6xo1J

# AI Ball — Ball Trap Brain

A toy "ball trap" symbolic model: input characters are tokenized against an alphabet, turned into a ball trajectory through a ring of deflectors, and the deflectors the ball falls through ("holes") produce the output characters.

This notebook contains:
1. The tokenizer and the `Ball_Trap_Brain` model itself
2. Fitness evaluation utilities
3. A genetic algorithm (`main`) that evolves a single brain
4. A straight-through Gumbel-Softmax training method (`Ball_Trap_Brain.st_train`) that trains a brain with gradient descent by sampling each discrete choice with Gumbel-Softmax and backpropagating through the straight-through estimator, instead of the genetic algorithm's mutate-and-select loop
5. A demo cell that trains a brain with `st_train` and evaluates it with the real, discrete `compute_input`

## Imports & tokenizer


In [ ]:

import random
import math
import copy

def TokenizeInput(input, alphabet):
    input_str = str(input)
    matches = []

    for order, item in enumerate(alphabet):
        item_str = str(item)
        if not item_str:
            continue

        start_pos = 0
        while True:
            index = input_str.find(item_str, start_pos)
            if index == -1:
                break
            matches.append((index, -len(item_str), order, item))
            start_pos = index + 1

    matches.sort(key=lambda match: (match[0], match[1], match[2]))

    non_overlapping_items = []
    last_end_index = 0
    for match in matches:
        start_index = match[0]
        length = -match[1]
        item = match[3]
        if start_index >= last_end_index:
            non_overlapping_items.append(item)
            last_end_index = start_index + length

    return non_overlapping_items


## `Ball_Trap_Brain`

Includes the original genetic-algorithm-oriented methods (`clone`, `Mutate`, `shift_alphabet`, ...) plus a gradient-based trainer built on straight-through Gumbel-Softmax:
- `st_train(...)`: trains the brain with gradient descent. Every discrete choice in the simulation (which deflector to route to, hole vs. bounce, which symbol, which direction, which start position) is sampled with Gumbel-Softmax and hardened to a one-hot vector on every forward pass, so the forward pass runs the same single, exact trajectory the discrete brain would run, while gradients flow back through the soft distribution via the straight-through estimator. `tau` anneals from `tau_start` to `tau_end` over training.
- `commit_st()`: writes the current straight-through parameters (argmax/rounded) into the brain's discrete `deflectors_list`, `alphabet_directions`, and `alphabet_start_positions`.
- `evaluate_st(test_cases, ratio=False)`: commits the current parameters onto a cloned brain and scores it with the real, discrete `compute_input`, without mutating the original brain.


In [ ]:

import torch
import torch.nn.functional as F


def _st_round(x):
    return x + (torch.round(x) - x).detach()


def _st_step(x, slope=4.0):
    hard = (x > 0).float()
    soft = torch.sigmoid(x * slope)
    return hard.detach() - soft.detach() + soft


class Ball_Trap_Brain:
    def __init__(self, deflectors, bounce, alphabet, output, holes_multiplier=1, direction_range=10):
        self.output = output
        self.holes_multiplier = holes_multiplier
        self.ball_energy = bounce * 2
        self.alphabet = alphabet
        self.direction_range = direction_range
        self.alphabet_directions = self.choose_alphabet_direction(len(alphabet))
        self.alphabet_start_positions = self.start_positions(len(alphabet), deflectors)
        self.deflectors_list = [[random.randint(0, deflectors - 1), bounce] for i in range(deflectors)]
        self.set_holes(output, deflectors, self.holes_multiplier)

    def set_holes(self, output, deflectors, holes_multiplier):
        for i in range(len(output)):
            for j in range(holes_multiplier):
                self.deflectors_list[random.randint(0, deflectors - 1)][1] = output[i]

    def choose_alphabet_direction(self, alphabet):
        return [random.randint(-self.direction_range, self.direction_range) for _ in range(alphabet)]

    def start_positions(self, alphabet, deflectors):
        return [random.randint(0, deflectors - 1) for i in range(alphabet)]

    def compute_input(self, input):
        directions = []
        positions = []
        travel = []
        input_arr = TokenizeInput(input, self.alphabet)
        for i in input_arr:
            if i in self.alphabet:
                index = self.alphabet.index(i)
                directions.append(self.alphabet_directions[index])
                positions.append(self.alphabet_start_positions[index])
        for i in range(len(directions)):
            travel.append(self.travel(copy.copy(directions), positions[i], i))
        return self.ricochet(travel, 10, self.ball_energy)

    def ricochet(self, travel, cap, ball_energy):
        output = []
        dl = self.deflectors_list
        n = len(dl)
        for t in travel:
            new_idx = t % n
            cap_idx = 0
            be = ball_energy
            while be > 0 and cap_idx < cap:
                dest, val = dl[new_idx]
                distance = abs(dest - new_idx)
                if not isinstance(val, int):
                    output.append(val)
                    be += 1
                    be -= distance
                    new_idx = dest
                    cap_idx += 1
                    continue
                be += val
                be -= distance
                new_idx = dest
                cap_idx += 1
        return output

    def travel(self, directions, position, index):
        total = 0
        weight_sum = 0
        for j, d in enumerate(directions):
            if j == index:
                continue
            weight = j + 1
            total += d * weight
            weight_sum += weight
        if weight_sum > 0:
            total = round(total / weight_sum)
        position += total
        return position

    def display_deflectors(self):
        for i, deflect in enumerate(self.deflectors_list):
            print(f"Deflector {i}: {deflect}")

    def clone(self):
        new_brain = Ball_Trap_Brain(1, self.ball_energy // 2, self.alphabet, self.output,
                                     self.holes_multiplier, self.direction_range)
        new_brain.deflectors_list = [[d, v] for d, v in self.deflectors_list]
        new_brain.alphabet_directions = self.alphabet_directions[:]
        new_brain.alphabet_start_positions = self.alphabet_start_positions[:]
        return new_brain

    def shift_alphabet(self):
        random.shuffle(self.alphabet)
        for i in range(len(self.deflectors_list)):
            if self.deflectors_list[i][1] in self.output:
                idx = self.output.index(self.deflectors_list[i][1])
                self.deflectors_list[i][1] = self.alphabet[idx]

    def Mutate(self, individual_probability=0.1, probability=0.05):
        if random.random() < individual_probability:
            for i in range(len(self.deflectors_list)):
                if random.random() < probability:
                    self.deflectors_list[i][0] = random.randint(0, len(self.deflectors_list) - 1)
            for i in range(len(self.deflectors_list)):
                if random.random() < probability:
                    self.deflectors_list[random.randint(0, len(self.deflectors_list) - 1)][1] = random.choice(self.output)
            for i in range(len(self.alphabet_directions)):
                if random.random() < probability:
                    self.alphabet_directions[i] = random.randint(-self.direction_range, self.direction_range)
            for i in range(len(self.alphabet_start_positions)):
                if random.random() < probability:
                    self.alphabet_start_positions[i] = random.randint(0, len(self.deflectors_list) - 1)
            if random.random() < probability:
                self.set_holes(self.output, len(self.deflectors_list), self.holes_multiplier)
        return self

    def _init_st_params(self, device=None):
        if device is None:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
        D = len(self.deflectors_list)
        A = len(self.alphabet)
        K = len(self.output)
        V = self.direction_range
        n_dirs = 2 * V + 1
        output_index = {sym: k for k, sym in enumerate(self.output)}
        CONF = 6.0
        dest_logits = torch.zeros(D, D, device=device)
        hole_logits = torch.zeros(D, 2, device=device)
        symbol_logits = torch.zeros(D, K, device=device)
        bounce_val = torch.zeros(D, device=device)
        direction_logits = torch.zeros(A, n_dirs, device=device)
        start_logits = torch.zeros(A, D, device=device)
        for i, (dest, val) in enumerate(self.deflectors_list):
            dest_logits[i, dest] = CONF
            if isinstance(val, int):
                hole_logits[i, 0] = CONF
                bounce_val[i] = float(val)
            else:
                hole_logits[i, 1] = CONF
                symbol_logits[i, output_index[val]] = CONF
        for i, d in enumerate(self.alphabet_directions):
            idx = max(0, min(n_dirs - 1, int(d) + V))
            direction_logits[i, idx] = CONF
        for i, p in enumerate(self.alphabet_start_positions):
            start_logits[i, p] = CONF
        for p in (dest_logits, hole_logits, symbol_logits, bounce_val, direction_logits, start_logits):
            p.requires_grad_(True)
        self._s_dest_logits = dest_logits
        self._s_hole_logits = hole_logits
        self._s_symbol_logits = symbol_logits
        self._s_bounce_val = bounce_val
        self._s_direction_logits = direction_logits
        self._s_start_logits = start_logits
        self._s_meta = dict(D=D, A=A, K=K, V=V, n_dirs=n_dirs, output_index=output_index, device=device)

    def _st_entry_state(self, travel_i, positions, D, tau):
        diff = positions - travel_i
        diff = torch.remainder(diff + D / 2, D) - D / 2
        logits = -(diff ** 2) / max(tau, 1e-3)
        soft = F.softmax(logits, dim=0)
        idx = torch.remainder(_st_round(travel_i), D).long()
        hard = F.one_hot(idx, D).float()
        return hard.detach() - soft.detach() + soft

    def _st_simulate(self, indices, direction_values, positions, tau, cap):
        m = self._s_meta
        D, K = m['D'], m['K']
        device = m['device']

        dir_onehot_all = F.gumbel_softmax(self._s_direction_logits, tau=tau, hard=True)
        start_onehot_all = F.gumbel_softmax(self._s_start_logits, tau=tau, hard=True)
        dir_values_all = dir_onehot_all @ direction_values
        start_values_all = start_onehot_all @ positions

        idx_t = torch.tensor(indices, dtype=torch.long, device=device)
        dirs = dir_values_all[idx_t]
        starts = start_values_all[idx_t]

        L = len(indices)
        pos_weights = torch.arange(1, L + 1, dtype=torch.float32, device=device)
        weighted = dirs * pos_weights
        total = weighted.sum()
        total_w = pos_weights.sum()

        out_symbols = []
        ever_fired_list = []
        extra_fired_list = []

        for i in range(L):
            w_i = torch.clamp(total_w - pos_weights[i], min=1e-8)
            avg_i = (total - weighted[i]) / w_i
            travel_i = starts[i] + avg_i
            state = self._st_entry_state(travel_i, positions, D, tau)

            be = torch.tensor(float(self.ball_energy), device=device)
            alive = torch.tensor(1.0, device=device)
            not_fired_yet = torch.tensor(1.0, device=device)
            out_accum = torch.zeros(K, device=device)
            extra = torch.tensor(0.0, device=device)

            for _ in range(cap):
                active_dest_logits = state @ self._s_dest_logits
                next_state = F.gumbel_softmax(active_dest_logits, tau=tau, hard=True)
                active_hole_logits = state @ self._s_hole_logits
                hole_onehot = F.gumbel_softmax(active_hole_logits, tau=tau, hard=True)
                is_hole = hole_onehot[1]
                active_symbol_logits = state @ self._s_symbol_logits
                symbol_onehot = F.gumbel_softmax(active_symbol_logits, tau=tau, hard=True)
                active_bounce = state @ self._s_bounce_val
                cur_idx = state @ positions
                next_idx = next_state @ positions
                distance = torch.abs(next_idx - cur_idx)

                already = alive * (1.0 - not_fired_yet)
                extra = extra + already * is_hole

                fire_now = alive * not_fired_yet * is_hole
                out_accum = out_accum + fire_now * symbol_onehot
                not_fired_yet = not_fired_yet * (1.0 - is_hole)

                delta = is_hole * 1.0 + (1.0 - is_hole) * active_bounce - distance
                be = be + delta
                alive = alive * _st_step(be)
                state = next_state

            out_symbols.append(out_accum)
            ever_fired_list.append(1.0 - not_fired_yet)
            extra_fired_list.append(extra)

        return out_symbols, ever_fired_list, extra_fired_list

    def st_train(self, test_cases, epochs=200, batch_size=16, tau_start=1.5, tau_end=0.2,
                 lr=0.05, cap=10, repeat_penalty_weight=0.1, blank_loss_weight=1.0,
                 commit=True, device=None, verbose=True):
        if not hasattr(self, '_s_meta'):
            self._init_st_params(device=device)
        m = self._s_meta
        K = m['K']
        output_index = m['output_index']
        device = m['device']
        params = [self._s_dest_logits, self._s_hole_logits, self._s_symbol_logits,
                  self._s_bounce_val, self._s_direction_logits, self._s_start_logits]
        optimizer = torch.optim.Adam(params, lr=lr)
        direction_values = torch.arange(-m['V'], m['V'] + 1, dtype=torch.float32, device=device)
        positions = torch.arange(m['D'], dtype=torch.float32, device=device)

        loss_history, acc_history = [], []

        for epoch in range(epochs):
            frac = epoch / max(1, epochs - 1)
            tau = tau_start + (tau_end - tau_start) * frac
            shuffled = list(test_cases)
            random.shuffle(shuffled)
            n_batches = max(1, (len(shuffled) + batch_size - 1) // batch_size)

            epoch_loss, epoch_correct, epoch_count = 0.0, 0, 0

            for b in range(n_batches):
                batch = shuffled[b * batch_size:(b + 1) * batch_size]
                if not batch:
                    continue
                optimizer.zero_grad()
                fire_losses, blank_losses, repeat_losses = [], [], []
                correct, count = 0, 0

                for inp, target in batch:
                    tokens = TokenizeInput(inp, self.alphabet)
                    indices = [self.alphabet.index(t) for t in tokens if t in self.alphabet]
                    if not indices:
                        continue
                    target_indices = [output_index[ch] for ch in target if ch in output_index]
                    L = len(indices)
                    n = min(L, len(target_indices))
                    out_offset = L - n
                    tgt_offset = len(target_indices) - n
                    target_classes = [K] * out_offset + [
                        target_indices[tgt_offset + (i - out_offset)] for i in range(out_offset, L)
                    ]

                    out_symbols, ever_fired, extra_fired = self._st_simulate(
                        indices, direction_values, positions, tau, cap
                    )

                    for i in range(L):
                        tc = target_classes[i]
                        if tc == K:
                            prob = 1.0 - ever_fired[i]
                            blank_losses.append(-torch.log(prob + 1e-8))
                        else:
                            prob = out_symbols[i][tc]
                            fire_losses.append(-torch.log(prob + 1e-8))
                        pred_fire = ever_fired[i].item() >= 0.5
                        pred_symbol = int(torch.argmax(out_symbols[i]).item()) if pred_fire else K
                        if pred_symbol == tc:
                            correct += 1
                        count += 1
                        repeat_losses.append(extra_fired[i])

                if count == 0:
                    continue

                fire_loss = torch.stack(fire_losses).mean() if fire_losses else torch.tensor(0.0, device=device)
                blank_loss = torch.stack(blank_losses).mean() if blank_losses else torch.tensor(0.0, device=device)
                task_loss = fire_loss + blank_loss_weight * blank_loss
                if repeat_losses and repeat_penalty_weight:
                    task_loss = task_loss + repeat_penalty_weight * torch.stack(repeat_losses).mean()

                task_loss.backward()
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                optimizer.step()

                epoch_loss += task_loss.item() * count
                epoch_correct += correct
                epoch_count += count

            if epoch_count == 0:
                continue
            avg_loss = epoch_loss / epoch_count
            avg_acc = epoch_correct / epoch_count
            loss_history.append(avg_loss)
            acc_history.append(avg_acc)
            if verbose and (epoch % max(1, epochs // 20) == 0 or epoch == epochs - 1):
                print(f"[st] epoch {epoch}: loss={avg_loss:.4f} acc={avg_acc:.3f} tau={tau:.3f}")

        if commit:
            self.commit_st()

        return loss_history, acc_history

    def commit_st(self, target=None):
        if target is None:
            target = self
        if not hasattr(self, '_s_meta'):
            self._init_st_params()
        m = self._s_meta
        with torch.no_grad():
            for i in range(m['D']):
                dest = int(torch.argmax(self._s_dest_logits[i]).item())
                if torch.argmax(self._s_hole_logits[i]).item() == 1:
                    val = self.output[int(torch.argmax(self._s_symbol_logits[i]).item())]
                else:
                    val = int(round(self._s_bounce_val[i].item()))
                target.deflectors_list[i][0] = dest
                target.deflectors_list[i][1] = val
            for i in range(m['A']):
                dir_idx = int(torch.argmax(self._s_direction_logits[i]).item())
                target.alphabet_directions[i] = int(dir_idx - m['V'])
                target.alphabet_start_positions[i] = int(torch.argmax(self._s_start_logits[i]).item())
        return target

    def evaluate_st(self, test_cases, ratio=False):
        temp_brain = self.clone()
        self.commit_st(target=temp_brain)
        return sum(evaluate_performance(inp, ans, temp_brain, ratio) for inp, ans in test_cases) / len(test_cases)


## Test case generation


In [ ]:

import random
import string

X = "The quick brown fox jumps over the lazy dog while the genetic algorithm and the straight-through Gumbel-Softmax trainer both try to teach the ball trap brain to predict the next character."

def generate_test_cases(amount, number, indiv=True):
    test_cases = []
    if indiv == True:
        test_cases = [(f'1+{i}', f"{random.randint(0, 1000)}") for i in range(amount)]
    else:
        for i in range(amount):
            a = []
            if i != len(X) - 1:
                b = X[i + 1]
            for j in range(i + 1):
                a.append(X[j])
                if len(a) > 12:
                    a.pop(0)
                s = ''.join(a)
            test_cases.append((s, b))

    return test_cases


## Fitness evaluation

Fitness evaluation module for Ball_Trap_Brain genetic algorithm.

This module contains functions to evaluate the performance of individual brains
and entire populations against test cases using string similarity metrics.


In [ ]:

import difflib
import math


def evaluate_performance(input_val, answer, brain, ratio=False):

    output = brain.compute_input(input_val)
    if ratio:
        return difflib.SequenceMatcher(None, ''.join(output), answer).ratio()
    else:
      return 1 if ''.join(output) == answer else 0


import concurrent.futures

def eval_single_brain(args):
    individual, test_cases, ratio = args
    avg_fitness = sum(
        evaluate_performance(inp, ans, individual, ratio) for inp, ans in test_cases
    ) / len(test_cases)
    return avg_fitness

def evaluate_population_parallel(population, test_cases, ratio=False):
    args = [(ind, test_cases, ratio) for ind in population]

    with concurrent.futures.ProcessPoolExecutor() as executor:
        results = list(executor.map(eval_single_brain, args))

    best_fitness = max(results)
    best_brain = population[results.index(best_fitness)]
    return best_fitness, best_brain


In [ ]:
def get_continuous_combinations(s):
    result = []
    n = len(s)
    for i in range(n):
        for j in range(i + 1, i + 5):
          if s[i:j] not in result:
            result.append(s[i:j])
    return result


## Genetic algorithm (`main`)

Genetic algorithm to evolve Ball_Trap_Brain instances to produce desired outputs.

This module runs a genetic algorithm that evolves a population of brains over
multiple generations. Each brain is evaluated against test cases, and the best
performer is mutated to create the next generation.


In [ ]:

string = "This module runs a genetic algorithm that evolves a population of brains over multiple generations. Each brain is evaluated against test cases, and the best performer is mutated to create the next generation."

import copy
import difflib
import random
import shelve

alphabet = get_continuous_combinations(X)
output = list("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ`_*(),.:[]-/|\"")

def main(mini_batch, alphabet, generations=500, pop_size=100, brain=Ball_Trap_Brain(1000, 4, alphabet, output, 2), ratio=False):
    """
    Run the genetic algorithm for evolving Ball_Trap_Brain instances.

    Returns:
        Ball_Trap_Brain: The best brain found after evolution.
    """
    population_size = pop_size

    population = [brain.clone().Mutate(1.0, 0.025) for _ in range(population_size)]
    population.append(brain)
    best_individual = None
    mini_batch_0 = copy.copy(mini_batch)
    highest_score = -1
    for generation in range(generations):

        best_fitness, best_individual = evaluate_population_parallel(population,mini_batch, ratio)

        if best_fitness >= 1:
          print(f"Generation {generation}: Solution found!")
          return best_individual

        new_population = [best_individual.clone().Mutate(1.0, 0.025) for _ in range(population_size)]
        population = new_population
        population.append(best_individual)
        print(f"Generation {generation}: Best fitness = {best_fitness}")
    return best_individual

training_1 = generate_test_cases(amount=100, number='1', indiv=True)


## Straight-through Gumbel-Softmax training — demo

Trains a brain with `st_train` (requires `torch`; installed via `pip install torch` if missing), then commits the trained parameters with `commit_st()` and evaluates the real, discrete `compute_input` against held-out test cases.


In [ ]:

import random
random.seed(0)

demo_brain = Ball_Trap_Brain(100, 1, output, output, 1, direction_range=70)
cases = generate_test_cases(amount=10, number='1', indiv=False)
random.shuffle(cases)
share = 0.8
demo_cases = cases[:int(share * len(cases))]
demo_test = cases[int(share * len(cases)):]

loss_history, acc_history = demo_brain.st_train(
    demo_cases,
    epochs=1500,
    batch_size=16,
    tau_start=1.5,
    tau_end=0.01,
    lr=0.1,
    commit=True,
    verbose=True,
)

print('token_acc[0] ->', acc_history[0], ' token_acc[-1] ->', acc_history[-1])
print()
print('=== discrete evaluation after commit_st() ===')
exact = sum(evaluate_performance(inp, ans, demo_brain, False) for inp, ans in demo_test) / len(demo_test)
ratio = sum(evaluate_performance(inp, ans, demo_brain, True) for inp, ans in demo_test) / len(demo_test)
print('exact-match:', exact)
print('ratio:', ratio)


[st] epoch 0: loss=36.2062 acc=0.028 tau=1.500
[st] epoch 75: loss=35.5710 acc=0.056 tau=1.425
[st] epoch 150: loss=20.9615 acc=0.694 tau=1.351
[st] epoch 225: loss=18.4207 acc=0.806 tau=1.276
[st] epoch 300: loss=15.7892 acc=0.833 tau=1.202
[st] epoch 375: loss=15.7892 acc=0.833 tau=1.127
[st] epoch 450: loss=17.0595 acc=0.778 tau=1.053
[st] epoch 525: loss=15.7892 acc=0.833 tau=0.978
[st] epoch 600: loss=18.4207 acc=0.806 tau=0.904
[st] epoch 675: loss=18.4207 acc=0.806 tau=0.829
[st] epoch 750: loss=18.4207 acc=0.806 tau=0.755
[st] epoch 825: loss=18.4207 acc=0.806 tau=0.680
[st] epoch 900: loss=16.4327 acc=0.806 tau=0.605
[st] epoch 975: loss=15.0799 acc=0.778 tau=0.531
[st] epoch 1050: loss=13.1743 acc=0.861 tau=0.456
[st] epoch 1125: loss=10.5455 acc=0.889 tau=0.382
[st] epoch 1200: loss=10.5317 acc=0.889 tau=0.307
[st] epoch 1275: loss=10.5483 acc=0.889 tau=0.233
[st] epoch 1350: loss=10.5317 acc=0.889 tau=0.158
[st] epoch 1425: loss=10.5261 acc=0.889 tau=0.084
[st] epoch 1499: 